# NB04b — Classifier Atribut Cahaya

Notebook ini melatih classifier multi-label dari piksel crop kendaraan. YOLO hanya mendeteksi kendaraan; classifier ini menentukan atribut `silau` dan `berwarna`, yang dapat aktif bersamaan.

Training dihentikan jika split `val` kosong atau dukungan atribut terlalu kecil.

In [ ]:
import json
import os
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RISET_ROOT = Path(os.environ.get("RISET_PROJECT_ROOT", Path.cwd())).resolve()
if RISET_ROOT.name not in {"Riset-HampirMatang", "NewStyle"}:
    raise RuntimeError(
        "Jalankan notebook dari root proyek atau set RISET_PROJECT_ROOT."
    )
sys.path.insert(0, str(RISET_ROOT))
sys.path.insert(0, str(RISET_ROOT / "builders"))
from attribute_classifier import (
    ATTRIBUTE_NAMES,
    FEATURE_SIZE,
    AttributeMLP,
    save_classifier,
)
from train_attribute_classifier import load_examples

SEED = 20261102
EPOCHS = 150
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
MIN_TRAIN_PER_ATTRIBUTE = 40
MIN_VAL_PER_ATTRIBUTE = 10
MODEL_DIR = RISET_ROOT / "output" / "attribute_classifier"
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## Data dan validasi split

Fitur dibuat dari statistik RGB/HSV, rasio piksel terang, rasio piksel bersaturasi, dan histogram hue. Split mengikuti kamera, bukan random per box, agar frame dari kamera yang sama tidak bocor ke validasi.

In [ ]:
features, labels, splits = load_examples()
train_mask = np.asarray([split == "train" for split in splits])
val_mask = np.asarray([split == "val" for split in splits])
if not train_mask.any() or not val_mask.any():
    raise RuntimeError(
        "Classifier membutuhkan gold box pada train dan val; val saat ini kosong."
    )
train_counts = labels[train_mask].sum(axis=0)
val_counts = labels[val_mask].sum(axis=0)
if np.any(train_counts < MIN_TRAIN_PER_ATTRIBUTE) or np.any(
    val_counts < MIN_VAL_PER_ATTRIBUTE
):
    raise RuntimeError(
        f"Gold atribut terlalu tipis: train={train_counts.tolist()} val={val_counts.tolist()}"
    )
display(
    {
        "examples_train": int(train_mask.sum()),
        "examples_val": int(val_mask.sum()),
        "attributes": list(ATTRIBUTE_NAMES),
        "positive_train": train_counts.tolist(),
        "positive_val": val_counts.tolist(),
    }
)

{'examples_train': 835,
 'examples_val': 710,
 'attributes': ['silau', 'berwarna'],
 'positive_train': [460.0, 128.0],
 'positive_val': [308.0, 121.0]}

## Training multi-label

Loss memakai positive weighting agar atribut langka tidak diabaikan. Threshold setiap atribut dipilih dari validation set berdasarkan F1, bukan dipatok sama untuk semua atribut.

In [ ]:
mean = features[train_mask].mean(axis=0)
std = features[train_mask].std(axis=0)
std[std < 1e-6] = 1.0
train_x = torch.from_numpy((features[train_mask] - mean) / std)
train_y = torch.from_numpy(labels[train_mask])
val_x = torch.from_numpy((features[val_mask] - mean) / std)
val_y = torch.from_numpy(labels[val_mask])
model = AttributeMLP(FEATURE_SIZE)
positive = train_y.sum(dim=0)
negative = len(train_y) - positive
loss_function = nn.BCEWithLogitsLoss(pos_weight=negative / torch.clamp(positive, min=1))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-3)
loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
best_state, best_loss = None, float("inf")
for _ in range(EPOCHS):
    model.train()
    for batch_x, batch_y in loader:
        optimizer.zero_grad()
        loss = loss_function(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        val_loss = float(loss_function(model(val_x), val_y))
    if val_loss < best_loss:
        best_loss = val_loss
        best_state = {
            key: value.detach().clone() for key, value in model.state_dict().items()
        }
model.load_state_dict(best_state)
with torch.no_grad():
    probabilities = torch.sigmoid(model(val_x)).numpy()
thresholds, metrics = {}, {}
for index, name in enumerate(ATTRIBUTE_NAMES):
    truth = val_y.numpy()[:, index] == 1
    candidates = np.arange(0.20, 0.81, 0.05)
    threshold = max(
        candidates,
        key=lambda candidate: (
            2
            * ((probabilities[:, index] >= candidate) & truth).sum()
            / max((probabilities[:, index] >= candidate).sum() + truth.sum(), 1),
            -candidate,
        ),
    )
    thresholds[name] = float(threshold)
    prediction = probabilities[:, index] >= threshold
    metrics[name] = {
        "support": int(truth.sum()),
        "precision": float((prediction & truth).sum() / max(prediction.sum(), 1)),
        "recall": float((prediction & truth).sum() / max(truth.sum(), 1)),
        "threshold": float(threshold),
    }
metadata = {
    "feature_version": "pixel-hsv-v1",
    "attribute_names": ATTRIBUTE_NAMES,
    "train_examples": int(train_mask.sum()),
    "val_examples": int(val_mask.sum()),
    "train_positive_counts": train_counts.tolist(),
    "val_positive_counts": val_counts.tolist(),
    "best_val_loss": best_loss,
    "metrics": metrics,
    "thresholds": thresholds,
}
MODEL_DIR.mkdir(parents=True, exist_ok=True)
save_classifier(MODEL_DIR / "best.pt", model, mean, std, metadata)
(MODEL_DIR / "metrics.json").write_text(
    json.dumps(metadata, indent=2), encoding="utf-8"
)
display(metadata)

{'feature_version': 'pixel-hsv-v1',
 'attribute_names': ('silau', 'berwarna'),
 'train_examples': 835,
 'val_examples': 710,
 'train_positive_counts': [460.0, 128.0],
 'val_positive_counts': [308.0, 121.0],
 'best_val_loss': 0.702642560005188,
 'metrics': {'silau': {'support': 308,
   'precision': 0.5414141414141415,
   'recall': 0.8701298701298701,
   'threshold': 0.49999999999999994},
  'berwarna': {'support': 121,
   'precision': 0.84,
   'recall': 0.6942148760330579,
   'threshold': 0.44999999999999996}},
 'thresholds': {'silau': 0.49999999999999994, 'berwarna': 0.44999999999999996}}

## Handoff

NB06 memuat `output/attribute_classifier/best.pt`. Jika notebook ini belum berhasil dijalankan, NB06 harus berhenti dan tidak boleh menerbitkan angka kondisi lampu.